# Module 04 — Building Neural Networks: From Scratch to `nn.Module`

**Prerequisites:** Module 03 (Autograd and Computational Graphs)
**Time:** ~70 minutes

## Learning Objectives

- Implement a linear layer manually as `y = x @ W + b`, and train it using raw autograd.
- Explain what `nn.Linear` abstracts away, having just built it by hand.
- Build a multi-layer network using `nn.Module` and explain parameter registration.
- Explain the role of activation functions and why stacking linear layers alone is not enough.


In [1]:
import torch
import torch.nn as nn


## Step 1: A Linear Layer, By Hand

Nearly every neural network is built from layers that compute $y = xW + b$ (a **linear transformation**), followed by a nonlinear **activation function**. Let's implement the linear part manually, so `nn.Linear` never feels like a black box.

We'll build a toy regression problem: predict `y` from `x` where the true relationship is `y = 3x + 2` (plus noise), and see if gradient descent can recover something close to `weight=3, bias=2` using nothing but tensors and autograd.


In [2]:
torch.manual_seed(0)

# Generate synthetic data: y = 3x + 2 + small noise
x = torch.linspace(-5, 5, 100).unsqueeze(1)          # shape (100, 1) -- 100 samples, 1 feature each
true_weight, true_bias = 3.0, 2.0
y = true_weight * x + true_bias + torch.randn_like(x) * 0.5   # add noise so it's not a perfectly clean line

print("x shape:", x.shape, " y shape:", y.shape)


x shape: torch.Size([100, 1])  y shape: torch.Size([100, 1])


We deliberately shaped `x` as `(100, 1)` rather than `(100,)` — a column of 100 single-feature samples — because that's the shape real feature data takes: `(num_samples, num_features)`. Get comfortable with this shape convention now; it appears everywhere from here on.


In [3]:
# Manually create the parameters. requires_grad=True: we want gradients w.r.t. these.
W = torch.randn(1, 1, requires_grad=True)   # shape (in_features=1, out_features=1)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.01
n_steps = 200

for step in range(n_steps):
    # Forward pass: the manual version of what nn.Linear does internally
    y_pred = x @ W + b

    # Loss: mean squared error, computed manually
    loss = ((y_pred - y) ** 2).mean()

    # Backward pass: autograd computes d(loss)/dW and d(loss)/db
    loss.backward()

    # Parameter update: manual gradient descent step.
    # torch.no_grad() because this update itself should NOT be tracked by autograd.
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    # Reset gradients -- required because they accumulate by default (Module 03)
    W.grad.zero_()
    b.grad.zero_()

    if step % 40 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f} | W {W.item():.3f} | b {b.item():.3f}")

print(f"\nFinal: W={W.item():.3f} (true={true_weight}), b={b.item():.3f} (true={true_bias})")


step   0 | loss 28.2370 | W 1.608 | b 0.040
step  40 | loss 1.0718 | W 2.999 | b 1.137
step  80 | loss 0.4232 | W 3.000 | b 1.626
step 120 | loss 0.2943 | W 3.000 | b 1.843
step 160 | loss 0.2687 | W 3.000 | b 1.941

Final: W=3.000 (true=3.0), b=1.983 (true=2.0)


Notice the recovered `W` and `b` land close to the true values `3.0` and `2.0` — gradient descent, using nothing but the autograd machinery from the previous module, found the underlying relationship in the noisy data. This entire loop — forward pass, loss, `.backward()`, manual update, zero gradients — *is* training, in its most stripped-down form. Every abstraction from here forward is just a more convenient way to write exactly this.


## Step 2: The Same Thing, With `nn.Linear`

`nn.Linear(in_features, out_features)` does exactly what we just wrote by hand: it creates a weight matrix and bias vector (shapes `(out_features, in_features)` and `(out_features,)`), both with `requires_grad=True` already set, and computes `x @ weight.T + bias` when called.


In [4]:
torch.manual_seed(0)

layer = nn.Linear(in_features=1, out_features=1)

print("weight shape:", layer.weight.shape)   # (1, 1)
print("bias shape:  ", layer.bias.shape)     # (1,)
print("weight requires_grad:", layer.weight.requires_grad)   # True automatically

# Using the layer is just calling it like a function:
sample_output = layer(x[:3])
print("\noutput for first 3 samples:", sample_output.squeeze())


weight shape: torch.Size([1, 1])
bias shape:   torch.Size([1])
weight requires_grad: True

output for first 3 samples: tensor([0.5739, 0.5731, 0.5724], grad_fn=<SqueezeBackward0>)


`layer.weight` and `layer.bias` are `nn.Parameter` objects — tensors that are automatically registered so that `layer.parameters()` (used to hand parameters to an optimizer) finds them without you doing anything manual. That registration is the main thing `nn.Linear` and `nn.Module` add on top of what you built by hand above: **automatic bookkeeping of every parameter in the model**, so that "grab every trainable number and hand it to the optimizer" is one line of code instead of a manually maintained list.


## Step 3: Why We Need Activation Functions

A linear layer alone can only represent straight-line (technically: linear) relationships. Stacking multiple linear layers *without* anything nonlinear between them doesn't help — the composition of any number of linear functions is still just one linear function.


In [5]:
# Two stacked linear layers with NO activation between them
stacked_linear = nn.Sequential(
    nn.Linear(1, 10),
    nn.Linear(10, 1),
)

# This is mathematically equivalent to a SINGLE linear layer -- verify by checking
# that the relationship between input and output is still a straight line.
test_x = torch.linspace(-5, 5, 5).unsqueeze(1)
test_y = stacked_linear(test_x).squeeze()
print("output:", test_y)
diffs = test_y[1:] - test_y[:-1]
print("differences between consecutive outputs (should be ~constant for a straight line):")
print(diffs)


output: tensor([ 1.7630,  0.8874,  0.0118, -0.8637, -1.7393],
       grad_fn=<SqueezeBackward0>)
differences between consecutive outputs (should be ~constant for a straight line):
tensor([-0.8756, -0.8756, -0.8756, -0.8756], grad_fn=<SubBackward0>)


The (nearly) constant differences confirm it: even with two layers and 10 "hidden" values in between, the overall input-output relationship is still just a straight line. Adding a **nonlinear activation function** between the layers is what allows the network to represent curves, and combinations of curves — which is what lets neural networks approximate essentially any function, given enough layers and width.

`nn.ReLU()` — Rectified Linear Unit — is the most common activation: it simply replaces negative values with zero, `max(0, x)`. Simple, but that one "kink" per unit is enough that stacking many of them lets a network approximate very complex, curved functions.


In [6]:
model_with_activation = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),        # the nonlinearity
    nn.Linear(10, 1),
)

test_y2 = model_with_activation(test_x).squeeze()
diffs2 = test_y2[1:] - test_y2[:-1]
print("differences between consecutive outputs (now NOT constant -- it's curved):")
print(diffs2)


differences between consecutive outputs (now NOT constant -- it's curved):
tensor([-0.2137, -0.3347, -1.0902, -1.0297], grad_fn=<SubBackward0>)


## Step 4: Building a Model as an `nn.Module` Subclass

For anything beyond the simplest cases, you'll define your model as a Python class inheriting from `nn.Module`. This is the pattern you'll use for the rest of the course.


In [7]:
class SimpleRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()   # MUST be called first -- see Module 00's warning about this
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # forward() defines what happens when you call model(x).
        # You never call forward() directly -- calling the model itself
        # (model(x)) triggers some internal nn.Module bookkeeping, then calls forward() for you.
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x


model = SimpleRegressor(input_dim=1, hidden_dim=16, output_dim=1)
print(model)


SimpleRegressor(
  (layer1): Linear(in_features=1, out_features=16, bias=True)
  (activation): ReLU()
  (layer2): Linear(in_features=16, out_features=1, bias=True)
)


**Why does assigning `self.layer1 = nn.Linear(...)` inside `__init__` matter?** Because `nn.Module.__init__()` (called via `super().__init__()`) sets up an internal registry. Whenever you assign an `nn.Module` (like `nn.Linear`) as an attribute of `self`, it gets automatically added to that registry. This is *why* `model.parameters()` below finds all the weights and biases, recursively, from every sub-layer, without you doing anything manual.


In [8]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params}")

for name, param in model.named_parameters():
    print(f"{name:20s} shape={tuple(param.shape)}")


Total trainable parameters: 49
layer1.weight        shape=(16, 1)
layer1.bias          shape=(16,)
layer2.weight        shape=(1, 16)
layer2.bias          shape=(1,)


## Shape Tracking Through the Model

Let's explicitly trace how the shape changes as data flows through this model — a habit worth building now, since it becomes essential once models get deeper.

```
Input
  |  shape (batch, 1)
  v
nn.Linear(1, 16)
  |  shape (batch, 16)
  v
nn.ReLU()
  |  shape (batch, 16)  -- ReLU doesn't change shape, only values
  v
nn.Linear(16, 1)
  |  shape (batch, 1)
  v
Output
```


In [9]:
batch = x[:8]   # take 8 samples, shape (8, 1)
print("input shape: ", batch.shape)

out1 = model.layer1(batch)
print("after layer1:", out1.shape)

out2 = model.activation(out1)
print("after ReLU:  ", out2.shape)   # unchanged from out1

out3 = model.layer2(out2)
print("after layer2:", out3.shape)

# Sanity check: this should match calling the model directly
direct = model(batch)
print("\nmatches model(batch)?", torch.allclose(out3, direct))


input shape:  torch.Size([8, 1])
after layer1: torch.Size([8, 16])
after ReLU:   torch.Size([8, 16])
after layer2: torch.Size([8, 1])

matches model(batch)? True


## 🐛 Debugging Challenge

The model below fails when given a batch of data. Find the bug before reading on.


In [10]:
class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1, 8)
        self.layer2 = nn.Linear(4, 1)   # bug is here

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

broken_model = BrokenModel()
try:
    output = broken_model(x[:4])
except RuntimeError as e:
    print("Error:", e)


Error: mat1 and mat2 shapes cannot be multiplied (4x8 and 4x1)


**Diagnosis:** `layer1` outputs 8 features (`nn.Linear(1, 8)`), but `layer2` expects an *input* of 4 features (`nn.Linear(4, 1)`). Every layer's output size must match the next layer's expected input size.

**Fix:** `layer2` should be `nn.Linear(8, 1)` to match `layer1`'s output.


## Exercises

🟢 **Beginner:** Build an `nn.Sequential` model with a `nn.Linear(1, 5)`, followed by `nn.ReLU()`, followed by `nn.Linear(5, 1)`. Print `model` and read off the layer shapes from the printed representation.

🟡 **Intermediate:** Build an `nn.Module` subclass called `TwoHiddenLayerNet` with two hidden layers of width 32 and 16 respectively (input dim and output dim as constructor arguments), each followed by `nn.ReLU()`. Print the total number of trainable parameters using `sum(p.numel() for p in model.parameters())`.

🔴 **Challenge:** Retrain the manual `W`/`b` linear regression from Step 1, but this time use `nn.Linear(1, 1)` and its `.parameters()` inside the training loop instead of manually tracked `W` and `b`. Update parameters manually (still using `torch.no_grad()` and `param.grad`, iterating over `layer.parameters()`) rather than using an optimizer — optimizers are introduced in the next module, and writing this update by hand first will make the optimizer's job click into place immediately.


In [11]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `super().__init__()`** — as covered in Module 00, this breaks `nn.Module`'s internal parameter registration.
- **Mismatched layer dimensions** — one layer's output size must equal the next layer's input size; PyTorch will not silently reshape this for you.
- **Stacking linear layers with no activation between them** — mathematically collapses to a single linear layer, wasting the extra parameters and depth.
- **Calling `model.forward(x)` directly instead of `model(x)`** — this skips internal `nn.Module` bookkeeping (hooks, mode-switching behavior you'll meet later) and should be avoided; always call the model, not `.forward()`, directly.

## Mental Model

An `nn.Module` is a *box that knows what's inside it*. When you put `nn.Linear` layers (or other modules) inside as attributes, the outer box automatically knows to include their parameters when someone asks "what are all your trainable numbers?" (`model.parameters()`). `forward()` is simply "what do you do when someone hands you an input" — the wiring diagram of how data flows through everything the box contains.

## Key Takeaways

- `y = x @ W + b`, computed and trained manually with autograd, is exactly what `nn.Linear` does — the abstraction adds parameter registration and convenience, not new math.
- Nonlinear activation functions (like `nn.ReLU`) are what let stacked layers represent curved, complex functions rather than collapsing to a single straight line.
- `nn.Module` subclasses register any `nn.Module` (or `nn.Parameter`) assigned as `self.<name>`, which is why `model.parameters()` finds everything automatically.
- Always track shapes as data flows through a model — mismatched layer dimensions are one of the most common errors you'll encounter.

## What's Next

**Module 05 — Loss Functions, Optimizers, and the Training Loop** replaces the manual gradient-descent update step you wrote by hand in this notebook with PyTorch's `torch.optim` optimizers, and introduces the standard loss functions used for classification and regression.

## Checklist

- [ ] I implemented a linear layer manually and trained it with raw autograd
- [ ] I can explain what `nn.Linear` and `nn.Module` add on top of raw tensors + autograd
- [ ] I can explain why activation functions are necessary between linear layers
- [ ] I can trace tensor shapes through a multi-layer model and diagnose a shape mismatch
